# Reparameterization Trick: 重参数技巧

## 背景
- 随机采样过程 $z \sim N(\mu, \sigma^2)$ 是不可导的，这个过程不可微，梯度无法传递

## 核心思想
- 将采样过程移到外部。不直接从 $N(\mu, \sigma^2)$ 中采样，而是引入一个独立的标准正态分布噪声  $\epsilon \sim N(0, 1)$ ，将 z 表示为参数的确定性函数 $z = \mu + \epsilon \cdot \sigma$
- 原来计算期望的梯度：$$\nabla_{\theta}\mathbb{E}_{z \sim P_{\theta}(z)}[f(z)]$$ 转化为了计算梯度的期望：$$\nabla_{\theta}\mathbb{E}_{\epsilon \sim P(\epsilon)}[f(g(\theta, \epsilon))] = \mathbb{E}_{\epsilon \sim P(\epsilon)}[\nabla_{\theta}f(g(\theta, \epsilon))]$$

## 优点
- 梯度可导
- 方差更小，更稳定可靠

## 问题
- 在离散情况下，$z = f(p,\epsilon)$ 永远是跳变的，导数 $\frac{\partial z}{\partial p}$ 几乎处处为0或者不存在，所以梯度无法反向传播。
- 需要 Score Function（REINFORCE）或 Gumbel-Softmax

### Score Function（REINFORCE）
- 不用对采样过程求导，而是利用：$$\nabla_{\theta}\mathbb{E}_{z \sim p_{\theta}(z)}[f(z)]$$ 数学推导：$$\nabla_{\theta}\mathbb{E}[f(z)] = E[f(z)\nabla_{\theta}\log{p_{theta}(z)}]$$
- 问题：方差巨大，训练不稳定，收敛慢

### Gumbel-Softmax：离散变量版的重采样技巧
- 原始Categorical采样不不可导，如：$$\pi = [0.1, 0.6, 0.3]$$
- 加入Gumbel噪声，采样：$$g_i = -\log{(-log{(u_i)})}$$ 其中：$$u_i \sim Uniform(0,1)$$ 然后计算：$$y_i = \frac{exp((\log{(\pi_i)} + g_i) / \tau)}{\Sigma_j exp((\log{(\pi_j)} + g_j) / \tau)}$$
- 这里的参数 $\tau$ 是temperature：当 $\tau \to 0$时，结果接近离散 [0,1,0]；当 $\tau$ 较大时，结果更平滑
- 由于整个过程：$\tau \to \log \to + \to / \to softmax$ 全部可导，所以梯度 $\frac{\partial z}{\partial \pi}$ 存在

不使用重参数：梯度无法正常传播

In [5]:
import torch

mu = torch.tensor(0.0, requires_grad=True)
sigma = torch.tensor(1.0, requires_grad=True)

z = torch.normal(mu, sigma)

loss = z**2

loss.backward()

print(mu.grad)
print(sigma.grad)
print(loss)

tensor(0.)
tensor(0.)
tensor(0.3236, grad_fn=<PowBackward0>)


使用重参数：梯度存在

In [6]:
import torch


mu = torch.tensor(0.0, requires_grad=True)
sigma = torch.tensor(1.0, requires_grad=True)

# 生成随机噪声
eps = torch.randn_like(mu)

# 重参数
z = mu + sigma * eps

loss = z**2
loss.backward()

print(mu.grad)
print(sigma.grad)
print(loss)

tensor(-2.0370)
tensor(2.0748)
tensor(1.0374, grad_fn=<PowBackward0>)


Gumbel Softmax

In [7]:
import torch
import torch.nn.functional as F

logits = torch.tensor([1.0,2.0,0.5], requires_grad=True)

z_1 = F.gumbel_softmax(logits, tau=1, hard=False)
print("hard=False: ", z_1)

z_2 = F.gumbel_softmax(logits, tau=1, hard=True)
print("hard=True: ", z_2)

hard=False:  tensor([0.1570, 0.4743, 0.3688], grad_fn=<SoftmaxBackward0>)
hard=True:  tensor([0., 1., 0.], grad_fn=<AddBackward0>)
